# Customer churn prediction model
## Objective
To develop a classification model that estimates the probability that a customer will churn.

The model will be used to:
1. identify customers with elevated churn risk
2. determine which customer characteristics contribute useful predictive information when considered simultaneously
3. support customer segmentation for targeted retention strategies

## 1. Setup

The cleaned customer dataset is loaded and prepared for predictive modeling. 
Scikit-learn will be used for processing, model training and evaluation.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [6]:
telco_clean_df = pd.read_csv("../../1_data/2_processed/telco_churn_clean.csv")

In [ ]:
telco_clean_df.head()

In [9]:
telco_clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        7043 non-null   str    
 1   gender             7043 non-null   str    
 2   senior_citizen     7043 non-null   int64  
 3   partner            7043 non-null   str    
 4   dependents         7043 non-null   str    
 5   tenure             7043 non-null   int64  
 6   phone_service      7043 non-null   str    
 7   multiple_lines     7043 non-null   str    
 8   internet_service   7043 non-null   str    
 9   online_security    7043 non-null   str    
 10  online_backup      7043 non-null   str    
 11  device_protection  7043 non-null   str    
 12  tech_support       7043 non-null   str    
 13  streaming_tv       7043 non-null   str    
 14  streaming_movies   7043 non-null   str    
 15  contract           7043 non-null   str    
 16  paperless_billing  7043 non-null   

In [ ]:
#converrting the target variable to binary
telco_clean_df["churn"] = telco_clean_df["churn"].map({'Yes': 1, 'No': 0})

In [12]:
telco_clean_df["churn"].value_counts(normalize=True)

churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

In [16]:
X = telco_clean_df.drop(columns=["customer_id", "churn"])
y = telco_clean_df["churn"]

In [17]:
X.head()

,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [18]:
y.head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

In [20]:
X.shape

(7043, 19)

In [23]:
y.shape

(7043,)

In [ ]:
#splitting the data into training and testing sets with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [26]:
y_train.value_counts(normalize=True)

churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

In [27]:
y_test.value_counts(normalize=True)

churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64

In [30]:
#defining the categorical and numerical features
numerical_features = [
    "senior_citizen",
    "tenure",
    "monthly_charges",
    "total_charges"
]

In [32]:
categorical_features = [
    col for col in X.columns if col not in numerical_features
]

In [35]:
numerical_features

['senior_citizen', 'tenure', 'monthly_charges', 'total_charges']

In [34]:
categorical_features

['gender',
 'partner',
 'dependents',
 'phone_service',
 'multiple_lines',
 'internet_service',
 'online_security',
 'online_backup',
 'device_protection',
 'tech_support',
 'streaming_tv',
 'streaming_movies',
 'contract',
 'paperless_billing',
 'payment_method']

In [38]:
#building the preprocessing pipeline
numeric_transformer = Pipeline(
    steps = [
        ("scaler", StandardScaler())
    ]
)